# Global Player Mobility in Football
**Notebook:** test.ipynb

**Project:** League-to-League Network Analysis


This notebook is structured and documented to support a CRISP-DM workflow: setup, audit, cleaning, mapping, edges, network metrics, hypothesis tests, visuals, and summary.

## 01 ? Setup
Imports, paths, and global settings.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "tu_data.db"
DB_PATH

WindowsPath('c:/Users/timgr/OneDrive/Dokumente/Interdis/data/tu_data.db')

## 02  Data Audit
Quick overview: tables, row counts, and a small sample.

In [2]:
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
tables

,name
0,tu_data_clubs
1,tu_data_marketvalues
2,tu_data_player
3,tu_data_squads
4,tu_data_transfers


In [3]:

table_names = tables['name'].tolist()
row_counts = []
for t in table_names:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t};", conn).iloc[0,0]
    row_counts.append({"table": t, "rows": cnt})
pd.DataFrame(row_counts).sort_values('rows', ascending=False)

,table,rows
1,tu_data_marketvalues,1186233
4,tu_data_transfers,925630
3,tu_data_squads,464552
2,tu_data_player,107321
0,tu_data_clubs,38229


In [4]:

transfers_sample = pd.read_sql("SELECT * FROM tu_data_transfers LIMIT 25;", conn)
transfers_sample

,id,details_playerId,details_date,typeDetails_type,transferSource_clubId,transferSource_competitionId,transferSource_countryId,transferSource_coachId,transferDestination_clubId,transferDestination_competitionId,transferDestination_countryId,transferDestination_coachId,details_contractUntilDate,details_age,details_fee_value,details_fee_currency,details_fee_compact_prefix,details_fee_compact_content,details_fee_compact_suffix,details_remainingContractPeriod_days
0,101736,10,10408.0,STANDARD,1768,,40,0,11451,,40,0,None,20,NaN,,,Free Transfer,,NaN
1,2489,10,10773.0,STANDARD,459,OL8,40,1981,151,RWS,40,51,None,21,NaN,,,Free Transfer,,NaN
2,23647,10,12600.0,STANDARD,2,L1,40,102,86,L1,40,224,None,26,5000000.0,EUR,€,5.00,M,NaN
3,412350,10,13695.0,STANDARD,86,L1,40,224,27,L1,40,92,None,29,15000000.0,EUR,€,15.00,M,NaN
4,581749,10,15156.0,STANDARD,27,L1,40,90,398,IT1,75,3158,2016-06-30T00:00:00+02:00,33,NaN,,,Free Transfer,,NaN
5,1480551,10,16983.0,STANDARD,398,IT1,75,0,123,,0,0,None,38,NaN,,,-,,NaN
6,1693574,100001,17198.0,STANDARD,288,AR1N,9,29746,209,AR1N,9,19885,2021-06-30T00:00:00+02:00,25,2500000.0,EUR,€,2.50,M,514.0
7,2204037,100001,17743.0,ACTIVE_LOAN_TRANSFER,209,AR1N,9,19885,2063,AR1N,9,4871,2019-06-30T00:00:00+02:00,27,90000.0,EUR,€,90.00,K,1065.0
8,2483940,100001,18077.0,RETURNED_FROM_PREVIOUS_LOAN,2063,AR1N,9,45679,209,AR1N,9,19885,2021-06-30T00:00:00+02:00,28,NaN,None,None,None,None,0.0
9,2584371,100001,18094.0,ACTIVE_LOAN_TRANSFER,209,AR1N,9,19885,333,AR1N,9,6890,2020-06-30T00:00:00+02:00,28,100000.0,EUR,€,100.00,K,714.0


## 02b ? Quick Exploration\n
Lightweight exploration to understand transfer coverage and key fields.

### Data Plausibility Checks
Plausibility pass for numeric/date fields before modeling assumptions.

In [ ]:
# Numeric plausibility (raw transfer table)
num_plaus = pd.read_sql(
    """
    SELECT
      COUNT(*) AS n_rows,
      MIN(details_date) AS min_details_date_raw,
      MAX(details_date) AS max_details_date_raw,
      MIN(details_fee_value) AS min_fee_value,
      MAX(details_fee_value) AS max_fee_value,
      AVG(details_fee_value) AS avg_fee_value,
      MIN(details_age) AS min_age,
      MAX(details_age) AS max_age,
      AVG(details_age) AS avg_age
    FROM tu_data_transfers
    """,
    conn
)
num_plaus


In [ ]:
# Detect the most plausible date encoding (days/sec/ms)
date_probe = pd.read_sql("SELECT details_date FROM tu_data_transfers WHERE details_date IS NOT NULL;", conn)
x = pd.to_numeric(date_probe['details_date'], errors='coerce')
cand_days = pd.to_datetime(x, unit='D', origin='1970-01-01', errors='coerce')
cand_sec = pd.to_datetime(x, unit='s', errors='coerce')
cand_ms = pd.to_datetime(x, unit='ms', errors='coerce')

lower = pd.Timestamp('1980-01-01')
upper = pd.Timestamp('2035-12-31')

def _score(dt):
    return int((dt.notna() & dt.between(lower, upper)).sum())

scores = pd.DataFrame({
    'encoding': ['days_since_1970', 'unix_seconds', 'unix_milliseconds'],
    'plausible_rows_1980_2035': [_score(cand_days), _score(cand_sec), _score(cand_ms)],
    'min_date': [cand_days.min(), cand_sec.min(), cand_ms.min()],
    'max_date': [cand_days.max(), cand_sec.max(), cand_ms.max()],
}).sort_values('plausible_rows_1980_2035', ascending=False)
scores


In [ ]:
# Parse using best encoding and inspect season tails
best_encoding = scores.iloc[0]['encoding']
if best_encoding == 'days_since_1970':
    parsed_dt = cand_days
elif best_encoding == 'unix_seconds':
    parsed_dt = cand_sec
else:
    parsed_dt = cand_ms

season_probe = parsed_dt.apply(lambda d: None if pd.isna(d) else (f"{d.year}/{d.year+1}" if d.month >= 7 else f"{d.year-1}/{d.year}"))

print('chosen_encoding:', best_encoding)
print('parsed_min_date:', parsed_dt.min())
print('parsed_max_date:', parsed_dt.max())
print('latest seasons:')
display(season_probe.value_counts().sort_index().tail(10))


In [ ]:
# Inspect future-dated transfers (explain unexpected future seasons)
future_cutoff = pd.Timestamp('2026-07-01')
tx_probe = pd.read_sql(
    """
    SELECT details_date, typeDetails_type, transferSource_competitionId, transferDestination_competitionId, details_fee_value
    FROM tu_data_transfers
    WHERE details_date IS NOT NULL
    """,
    conn
)
tx_probe['details_date_num'] = pd.to_numeric(tx_probe['details_date'], errors='coerce')
if best_encoding == 'days_since_1970':
    tx_probe['parsed_date'] = pd.to_datetime(tx_probe['details_date_num'], unit='D', origin='1970-01-01', errors='coerce')
elif best_encoding == 'unix_seconds':
    tx_probe['parsed_date'] = pd.to_datetime(tx_probe['details_date_num'], unit='s', errors='coerce')
else:
    tx_probe['parsed_date'] = pd.to_datetime(tx_probe['details_date_num'], unit='ms', errors='coerce')

future_rows = tx_probe[tx_probe['parsed_date'] >= future_cutoff].copy()
print('future_rows_count:', len(future_rows))
display(future_rows['typeDetails_type'].value_counts().head(15))
future_rows.sort_values('parsed_date').head(30)


### Data QA Decisions
Inspect outliers and define a reproducible quality filter for modeling.

In [ ]:
# Outlier counts in raw fields
qa_counts = pd.read_sql(
    """
    SELECT
      SUM(CASE WHEN details_date < 0 THEN 1 ELSE 0 END) AS n_negative_date,
      SUM(CASE WHEN details_age > 60 THEN 1 ELSE 0 END) AS n_age_gt_60,
      SUM(CASE WHEN details_age > 100 THEN 1 ELSE 0 END) AS n_age_gt_100
    FROM tu_data_transfers
    """,
    conn
)
qa_counts


In [ ]:
# Example extreme-age records
pd.read_sql(
    """
    SELECT details_playerId, details_date, details_age, typeDetails_type,
           transferSource_competitionId, transferDestination_competitionId
    FROM tu_data_transfers
    WHERE details_age > 100
    """,
    conn
)


In [ ]:
# Recommended analysis window / quality mask
# Tune these bounds if your final scope changes.
ANALYSIS_MIN_DATE = pd.Timestamp('1990-01-01')
ANALYSIS_MAX_DATE = pd.Timestamp('2025-06-30')
ANALYSIS_MIN_AGE = 14
ANALYSIS_MAX_AGE = 45

# Build a transfer-level QA frame for exploration only
qa_df = pd.read_sql(
    "SELECT details_date, details_age, typeDetails_type, details_fee_value FROM tu_data_transfers",
    conn
)
qa_num = pd.to_numeric(qa_df['details_date'], errors='coerce')
if best_encoding == 'days_since_1970':
    qa_df['parsed_date'] = pd.to_datetime(qa_num, unit='D', origin='1970-01-01', errors='coerce')
elif best_encoding == 'unix_seconds':
    qa_df['parsed_date'] = pd.to_datetime(qa_num, unit='s', errors='coerce')
else:
    qa_df['parsed_date'] = pd.to_datetime(qa_num, unit='ms', errors='coerce')

qa_df['in_analysis_window'] = qa_df['parsed_date'].between(ANALYSIS_MIN_DATE, ANALYSIS_MAX_DATE)
qa_df['age_plausible'] = qa_df['details_age'].between(ANALYSIS_MIN_AGE, ANALYSIS_MAX_AGE)
qa_df['keep_for_analysis'] = qa_df['in_analysis_window'] & qa_df['age_plausible']

qa_df[['in_analysis_window','age_plausible','keep_for_analysis']].mean().to_frame('share')


### Exploration Plots
Quick plots to document distribution shape and graph structure.

In [ ]:
import matplotlib.pyplot as plt

# Parsed date distribution (yearly)
year_counts = qa_df['parsed_date'].dropna().dt.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12, 4))
year_counts.plot(ax=ax)
ax.set_title('Transfer records by calendar year')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.show()


In [ ]:
# Age distribution (trimmed for readability)
age_plot = qa_df['details_age'].dropna()
age_plot = age_plot[(age_plot >= 10) & (age_plot <= 50)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(age_plot, bins=40)
ax.set_title('Age distribution (10-50)')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
plt.show()


In [ ]:
# Graph exploration: strength distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log1p(metrics['in_strength']), bins=40)
axes[0].set_title('log(1 + in_strength)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(metrics['out_strength']), bins=40)
axes[1].set_title('log(1 + out_strength)')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Graph exploration: community sizes
comm_sizes = metrics['community'].value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
comm_sizes.plot(kind='bar', ax=ax)
ax.set_title('Community sizes (number of leagues)')
ax.set_xlabel('Community ID')
ax.set_ylabel('Leagues')
plt.show()


In [5]:
# Check date field distribution (assumes Unix timestamp; verify)
df = pd.read_sql("SELECT details_date FROM tu_data_transfers WHERE details_date IS NOT NULL LIMIT 200000;", conn)
df['details_date'].describe()

count    200000.000000
mean      16697.196095
std        2218.490595
min      -21735.000000
25%       15279.000000
50%       16648.000000
75%       18444.000000
max       20970.000000
Name: details_date, dtype: float64

In [6]:
# Peek at distinct transfer types
pd.read_sql("SELECT typeDetails_type, COUNT(*) AS n FROM tu_data_transfers GROUP BY typeDetails_type ORDER BY n DESC LIMIT 30;", conn)

,typeDetails_type,n
0,STANDARD,672927
1,ACTIVE_LOAN_TRANSFER,126500
2,RETURNED_FROM_PREVIOUS_LOAN,126203


In [7]:
# Missingness snapshot for core fields
core_cols = [
    'details_playerId', 'details_date',
    'transferSource_competitionId', 'transferDestination_competitionId',
    'transferSource_countryId', 'transferDestination_countryId',
    'details_fee_value', 'typeDetails_type'
]
query = 'SELECT ' + ','.join([f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}_nulls" for c in core_cols]) + ' FROM tu_data_transfers'
pd.read_sql(query, conn)

,details_playerId_nulls,details_date_nulls,transferSource_competitionId_nulls,transferDestination_competitionId_nulls,transferSource_countryId_nulls,transferDestination_countryId_nulls,details_fee_value_nulls,typeDetails_type_nulls
0,0,308,0,0,0,0,879715,0


### Fee missingness check\n
Compare null fee values to compact content labels (e.g., Free Transfer).

In [ ]:
# How often are fee_value nulls labeled as Free Transfer / Loan / etc.?
transfers_fee = pd.read_sql(
    "SELECT details_fee_value, details_fee_compact_content, typeDetails_type FROM tu_data_transfers;",
    conn
)
fee_nulls = transfers_fee[transfers_fee['details_fee_value'].isna()]
fee_labels = (fee_nulls['details_fee_compact_content']
    .astype(str).str.strip().str.lower()
    .value_counts().head(30))
fee_labels


In [ ]:
# Share of fee nulls that are explicitly marked as free/loan/undisclosed
labels = fee_nulls['details_fee_compact_content'].astype(str).str.lower()
is_free = labels.str.contains('free', na=False)
is_loan = labels.str.contains('loan', na=False)
is_undisclosed = labels.str.contains('undisclosed|unknown|n/a', na=False)
pd.DataFrame({
    'fee_null_rows': [len(fee_nulls)],
    'free_share': [is_free.mean()],
    'loan_share': [is_loan.mean()],
    'undisclosed_share': [is_undisclosed.mean()],
})

### Fee content vs transfer type\n
Cross-tab of fee content labels by transfer type.

In [27]:
# Crosstab only for zero-fee transfers
fee_zero = transfers_fee[transfers_fee['details_fee_value'].fillna(-1) == 0]
fee_vs_type = pd.crosstab(
    fee_zero['typeDetails_type'].astype(str).str.lower(),
    fee_zero['details_fee_compact_content'].astype(str).str.lower(),
    dropna=False
)
fee_vs_type.iloc[:20, :20]


details_fee_compact_content,0.00
typeDetails_type,
active_loan_transfer,92
returned_from_previous_loan,70
standard,206


In [ ]:
# Row-normalized shares (percent) for zero-fee transfers
fee_vs_type_pct = fee_vs_type.div(fee_vs_type.sum(axis=1), axis=0) * 100
fee_vs_type_pct.iloc[:20, :20].round(2)

In [8]:
# Top source competitions
pd.read_sql("SELECT transferSource_competitionId, COUNT(*) AS n FROM tu_data_transfers GROUP BY transferSource_competitionId ORDER BY n DESC LIMIT 20;", conn)

,transferSource_competitionId,n
0,,297980
1,IT1,16912
2,BRA1,15167
3,IT2,13536
4,BRA2,12275
5,GB2,12104
6,GB3,10375
7,TR1,9702
8,GB1,9188
9,TR2,8988


In [9]:
# Top destination competitions
pd.read_sql("SELECT transferDestination_competitionId, COUNT(*) AS n FROM tu_data_transfers GROUP BY transferDestination_competitionId ORDER BY n DESC LIMIT 20;", conn)

,transferDestination_competitionId,n
0,,311128
1,IT1,16640
2,BRA1,13834
3,IT2,13228
4,BRA2,11939
5,GB2,11430
6,GB3,9986
7,TR1,9445
8,GB1,9097
9,TR2,8616


In [11]:
# Domestic vs international share (based on countryId fields)
pd.read_sql("SELECT SUM(CASE WHEN transferSource_countryId = transferDestination_countryId THEN 1 ELSE 0 END) AS domestic, SUM(CASE WHEN transferSource_countryId != transferDestination_countryId THEN 1 ELSE 0 END) AS international FROM tu_data_transfers WHERE transferSource_countryId IS NOT NULL AND transferDestination_countryId IS NOT NULL;", conn)


,domestic,international
0,605791,319839


## 03 ? Transfer Cleaning (stub)\n
Parse dates, normalize types, and create Season mapping.

In [12]:
# Transfer cleaning: parse date, normalize type, derive season
transfers = pd.read_sql("SELECT * FROM tu_data_transfers;", conn)

# Parse details_date with automatic format detection (days / seconds / milliseconds)
date_num = pd.to_numeric(transfers['details_date'], errors='coerce')
cand_days = pd.to_datetime(date_num, unit='D', origin='1970-01-01', errors='coerce')
cand_sec = pd.to_datetime(date_num, unit='s', errors='coerce')
cand_ms = pd.to_datetime(date_num, unit='ms', errors='coerce')

lower = pd.Timestamp('1980-01-01')
upper = pd.Timestamp('2035-12-31')

def _score(dt):
    ok = dt.notna() & dt.between(lower, upper)
    return int(ok.sum())

candidates = [(cand_days, _score(cand_days)), (cand_sec, _score(cand_sec)), (cand_ms, _score(cand_ms))]
transfers['details_date_parsed'] = sorted(candidates, key=lambda x: x[1], reverse=True)[0][0]
# Normalize transfer type (lowercase, strip)
transfers['transfer_type_norm'] = transfers['typeDetails_type'].astype(str).str.strip().str.lower()

# Basic loan flag heuristic
transfers['is_loan'] = transfers['transfer_type_norm'].str.contains('loan', na=False)

# Season mapping: if month >= July -> season = year/year+1 else year-1/year
def map_season(dt):
    if pd.isna(dt):
        return None
    year = dt.year
    if dt.month >= 7:
        return f"{year}/{year+1}"
    return f"{year-1}/{year}"

transfers['season'] = transfers['details_date_parsed'].apply(map_season)

# Quick sanity checks
transfers[['details_date','details_date_parsed','season','typeDetails_type','transfer_type_norm','is_loan']].head(10)

,details_date,details_date_parsed,season,typeDetails_type,transfer_type_norm,is_loan
0,10408.0,1970-01-01 02:53:28,1969/1970,STANDARD,standard,False
1,10773.0,1970-01-01 02:59:33,1969/1970,STANDARD,standard,False
2,12600.0,1970-01-01 03:30:00,1969/1970,STANDARD,standard,False
3,13695.0,1970-01-01 03:48:15,1969/1970,STANDARD,standard,False
4,15156.0,1970-01-01 04:12:36,1969/1970,STANDARD,standard,False
5,16983.0,1970-01-01 04:43:03,1969/1970,STANDARD,standard,False
6,17198.0,1970-01-01 04:46:38,1969/1970,STANDARD,standard,False
7,17743.0,1970-01-01 04:55:43,1969/1970,ACTIVE_LOAN_TRANSFER,active_loan_transfer,True
8,18077.0,1970-01-01 05:01:17,1969/1970,RETURNED_FROM_PREVIOUS_LOAN,returned_from_previous_loan,True
9,18094.0,1970-01-01 05:01:34,1969/1970,ACTIVE_LOAN_TRANSFER,active_loan_transfer,True


## 04 ? League/Competition Mapping (stub)\n
Map competitionId to league name, country, and tier.

In [ ]:
# Competition mapping: reuse existing CSV (do not rebuild here)
mapping_path = PROJECT_ROOT / 'data' / 'competition_mapping.csv'

if not mapping_path.exists():
    raise FileNotFoundError("competition_mapping.csv not found. Run scripts/competition_mapping.py locally first.")

comp_map = pd.read_csv(mapping_path)

# Ensure canonical columns exist
for col in ['canonical_competition_id','prior_league_names']:
    if col not in comp_map.columns:
        comp_map[col] = pd.NA

comp_map.head(10)

In [14]:
# Mapping completeness check
comp_map.isna().mean().sort_values(ascending=False)

error             0.813986
tier              0.202797
country_id        0.193007
league_name       0.186014
competition_id    0.000000
source_url        0.000000
dtype: float64

In [15]:
# Rows with missing key fields
missing = comp_map[comp_map[['league_name','country_id','tier']].isna().any(axis=1)]
missing.head(25)

,competition_id,league_name,country_id,tier,source_url,error
69,BBPO,Landespokal Brandenburg,40.0,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
71,BCP1,Campeonato Paulista,26.0,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
72,BCP2,Campeonato Paulista - Troféu Independência,26.0,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
97,BRA3,Campeonato Brasileiro Série C,NaN,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
100,BRCE,Campeonato Cearense,NaN,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
103,BRCR,Campeonato Carioca - Taça Rio,26.0,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
107,BRNE,Copa do Nordeste,26.0,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
109,BRPE,Campeonato Pernambucano,NaN,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
112,BRRS,Campeonato Gaúcho,26.0,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN
113,BRSC,Campeonato Catarinense,NaN,NaN,https://www.transfermarkt.de/jumplist/startsei...,NaN


In [ ]:
# Unique tiers overview
comp_map['tier'].value_counts(dropna=False).sort_index()

tier
1                    172
2                     95
3                    101
4                     92
5                     56
6                     45
Reserve               11
StateChampionship     12
Youth                101
NaN                   30
Name: count, dtype: int64

## 05 ? Edge Lists (stub)\n
Create league-to-league edges per season and aggregated.

In [ ]:
# Build edge lists (per season and aggregated)
# Join transfers with competition mapping for source and destination
comp_map_src = comp_map.rename(columns={
    'competition_id': 'transferSource_competitionId',
    'league_name': 'source_league',
    'country_id': 'source_country_id',
    'tier': 'source_tier',
    'canonical_competition_id': 'source_canonical_id',
})

comp_map_dst = comp_map.rename(columns={
    'competition_id': 'transferDestination_competitionId',
    'league_name': 'dest_league',
    'country_id': 'dest_country_id',
    'tier': 'dest_tier',
    'canonical_competition_id': 'dest_canonical_id',
})

edges_base = (transfers
    .merge(comp_map_src, on='transferSource_competitionId', how='left')
    .merge(comp_map_dst, on='transferDestination_competitionId', how='left')
)

# Use canonical IDs when present
edges_base['source_league_id'] = edges_base['source_canonical_id'].fillna(edges_base['transferSource_competitionId'])
edges_base['dest_league_id'] = edges_base['dest_canonical_id'].fillna(edges_base['transferDestination_competitionId'])

# Domestic vs international
edges_base['is_domestic'] = edges_base['transferSource_countryId'] == edges_base['transferDestination_countryId']

# Keep only rows with both leagues resolved
edges_base = edges_base[edges_base['source_league'].notna() & edges_base['dest_league'].notna()]

# Aggregate per season
edge_season = (edges_base
    .groupby(['season','source_league_id','dest_league_id','source_league','dest_league','source_country_id','dest_country_id','source_tier','dest_tier'], dropna=False)
    .agg(
        n_transfers=('details_playerId','count'),
        n_loans=('is_loan','sum'),
        fee_sum=('details_fee_value','sum'),
        fee_median=('details_fee_value','median'),
        domestic_share=('is_domestic','mean')
    )
    .reset_index()
)

# Aggregate overall (all seasons)
edge_all = (edges_base
    .groupby(['source_league_id','dest_league_id','source_league','dest_league','source_country_id','dest_country_id','source_tier','dest_tier'], dropna=False)
    .agg(
        n_transfers=('details_playerId','count'),
        n_loans=('is_loan','sum'),
        fee_sum=('details_fee_value','sum'),
        fee_median=('details_fee_value','median'),
        domestic_share=('is_domestic','mean')
    )
    .reset_index()
)

edge_season.head(10)

## 05b ? Graph Construction\n
Build directed graphs from the edge lists (aggregated + per season).

In [ ]:
import igraph as ig

# Build readable league labels for node ids
src_labels = edge_all[['source_league_id', 'source_league']].dropna().drop_duplicates()
dst_labels = edge_all[['dest_league_id', 'dest_league']].dropna().drop_duplicates()
src_labels.columns = ['league_id', 'league_name']
dst_labels.columns = ['league_id', 'league_name']
league_labels = (
    pd.concat([src_labels, dst_labels], ignore_index=True)
    .drop_duplicates(subset=['league_id'])
    .set_index('league_id')['league_name']
    .to_dict()
)

# Aggregated graph (all seasons)
agg_tuples = [(r.source_league_id, r.dest_league_id, float(r.n_transfers)) for r in edge_all.itertuples(index=False)]
G_all = ig.Graph.TupleList(agg_tuples, directed=True, weights=True)
G_all.es['distance'] = [1.0 / w if w > 0 else 1e9 for w in G_all.es['weight']]
G_all.vs['league_name'] = [league_labels.get(v['name'], v['name']) for v in G_all.vs]

# Per-season graphs
graphs_by_season = {}
for season, df in edge_season.groupby('season'):
    tuples = [(r.source_league_id, r.dest_league_id, float(r.n_transfers)) for r in df.itertuples(index=False)]
    g = ig.Graph.TupleList(tuples, directed=True, weights=True)
    g.es['distance'] = [1.0 / w if w > 0 else 1e9 for w in g.es['weight']]
    g.vs['league_name'] = [league_labels.get(v['name'], v['name']) for v in g.vs]
    graphs_by_season[season] = g

G_all.vcount(), G_all.ecount()


## 06 ? Network Metrics (starter)\n
Compute weighted in/out strength, degree, and betweenness for the aggregated graph.

In [ ]:
# Metrics from igraph (aggregated graph)
metrics = pd.DataFrame({
    'league': G_all.vs['name'],
    'league_name': G_all.vs['league_name'],
    'in_strength': G_all.strength(mode='in', weights='weight'),
    'out_strength': G_all.strength(mode='out', weights='weight'),
    'in_degree': G_all.degree(mode='in'),
    'out_degree': G_all.degree(mode='out'),
    # Betweenness uses distance, so we invert transfer weights in graph construction
    'betweenness': G_all.betweenness(directed=True, weights='distance'),
})

metrics.sort_values('in_strength', ascending=False).head(15)


## 06b ? Community Detection\n
Try Louvain/Leiden if available; fallback to a NetworkX method.

In [ ]:
# Community detection on aggregated graph (single method: Leiden)
import leidenalg

parts = leidenalg.find_partition(
    G_all,
    leidenalg.RBConfigurationVertexPartition,
    weights='weight',
    seed=42,
)

communities = {G_all.vs[idx]['name']: int(parts.membership[idx]) for idx in range(G_all.vcount())}
metrics['community'] = metrics['league'].map(communities)
'leiden'


## 06c - Community Summary
Quick profiling of detected communities and their key leagues.

In [ ]:
# Community sizes
community_sizes = (metrics.groupby('community', dropna=False)
    .agg(n_leagues=('league','nunique'))
    .reset_index()
    .sort_values('n_leagues', ascending=False))
community_sizes.head(20)


In [ ]:
# Top leagues per community by in_strength
community_top_in = (metrics.sort_values(['community','in_strength'], ascending=[True, False])
    .groupby('community', dropna=False)
    .head(5)
    [['community','league','league_name','in_strength','out_strength','betweenness']])
community_top_in.head(50)


## 07a - H1 Concentration
Test whether incoming transfer flows are concentrated in a small set of leagues.

In [ ]:
# Build league-level inflow from aggregated edges
inflow = (edge_all.groupby('dest_league_id', dropna=False)
    .agg(total_in_transfers=('n_transfers','sum'))
    .reset_index()
    .sort_values('total_in_transfers', ascending=False))

# Add readable labels
name_map = metrics.set_index('league')['league_name'].to_dict()
inflow['league_name'] = inflow['dest_league_id'].map(name_map).fillna(inflow['dest_league_id'])

inflow.head(20)


In [ ]:
# Concentration KPIs: Top-k shares + HHI
total_in = inflow['total_in_transfers'].sum()
top5_share = inflow.head(5)['total_in_transfers'].sum() / total_in if total_in > 0 else 0
top10_share = inflow.head(10)['total_in_transfers'].sum() / total_in if total_in > 0 else 0
shares = inflow['total_in_transfers'] / total_in if total_in > 0 else inflow['total_in_transfers']
hhi = (shares ** 2).sum()

pd.DataFrame({
    'total_in_transfers': [total_in],
    'top5_share': [top5_share],
    'top10_share': [top10_share],
    'hhi': [hhi],
}).round(4)


## 07b - H2 Bridges
Identify leagues with high betweenness and check whether mid-tier leagues are overrepresented.

In [ ]:
# League-level tier label (fallback to 'Unknown' if missing)
tier_map_src = edge_all[['source_league_id','source_tier']].dropna().drop_duplicates().rename(columns={'source_league_id':'league','source_tier':'tier'})
tier_map_dst = edge_all[['dest_league_id','dest_tier']].dropna().drop_duplicates().rename(columns={'dest_league_id':'league','dest_tier':'tier'})
tier_map = pd.concat([tier_map_src, tier_map_dst], ignore_index=True).drop_duplicates(subset=['league'])
tier_map_dict = tier_map.set_index('league')['tier'].to_dict()

metrics['tier'] = metrics['league'].map(tier_map_dict).fillna('Unknown')
metrics['tier_num'] = pd.to_numeric(metrics['tier'], errors='coerce')
# Mid-tier heuristic: numeric tier 2-4
metrics['is_mid_tier'] = metrics['tier_num'].between(2, 4, inclusive='both').fillna(False)

metrics[['league','league_name','tier','is_mid_tier','betweenness']].head()


In [ ]:
# Top bridge leagues by betweenness
bridge_rank = (metrics
    .sort_values('betweenness', ascending=False)
    [['league','league_name','tier','is_mid_tier','betweenness','in_strength','out_strength']])
bridge_rank.head(25)


In [ ]:
# H2 diagnostic: mid-tier share in top-k betweenness vs overall
k = 25
topk = bridge_rank.head(k)
mid_share_topk = topk['is_mid_tier'].mean() if len(topk) else 0
mid_share_all = metrics['is_mid_tier'].mean() if len(metrics) else 0

pd.DataFrame({
    'k': [k],
    'mid_share_topk_betweenness': [mid_share_topk],
    'mid_share_all_leagues': [mid_share_all],
    'lift_topk_vs_all': [mid_share_topk / mid_share_all if mid_share_all > 0 else None],
}).round(4)


## 07c - H3 Community Structure vs Null Model
Compare observed Leiden modularity against degree-preserving rewired baselines.

In [ ]:
# Observed modularity from current Leiden partition
observed_modularity = parts.quality()
observed_modularity


In [ ]:
# Null model: degree-preserving rewiring + Leiden modularity
import numpy as np

n_runs = 30
niter_factor = 10  # rewiring intensity relative to edge count
null_modularity = []

for run in range(n_runs):
    g_null = G_all.copy()
    n_rewire = int(max(1, g_null.ecount() * niter_factor))

    # Preferred API (new igraph):
    try:
        g_null.rewire(n=n_rewire, allowed_edge_types='simple')
    except TypeError:
        # Backward-compatible API
        g_null.rewire(n=n_rewire, mode='simple')

    # For null graphs, use unweighted Leiden to avoid version/weight edge-cases
    null_parts = leidenalg.find_partition(
        g_null,
        leidenalg.RBConfigurationVertexPartition,
        seed=42 + run,
    )
    null_modularity.append(null_parts.quality())

null_modularity = np.array(null_modularity)
pd.Series(null_modularity).describe()


In [ ]:
# H3 comparison table
null_mean = float(null_modularity.mean())
null_std = float(null_modularity.std(ddof=1)) if len(null_modularity) > 1 else np.nan
z_score = (observed_modularity - null_mean) / null_std if null_std and not np.isnan(null_std) else np.nan
p_one_sided = float((null_modularity >= observed_modularity).mean())

pd.DataFrame({
    'observed_modularity': [observed_modularity],
    'null_mean_modularity': [null_mean],
    'null_std_modularity': [null_std],
    'z_score_vs_null': [z_score],
    'p_value_one_sided': [p_one_sided],
}).round(6)


## 07d - H4 Lower-Tier Pathways
Assess lower-tier upward mobility and domestic vs cross-border patterns.

In [ ]:
# Prepare tier flow frame from transfer-level data
h4_df = edges_base.copy()
h4_df['source_tier_num'] = pd.to_numeric(h4_df['source_tier'], errors='coerce')
h4_df['dest_tier_num'] = pd.to_numeric(h4_df['dest_tier'], errors='coerce')
h4_df = h4_df[h4_df['source_tier_num'].notna() & h4_df['dest_tier_num'].notna()].copy()

h4_df['is_domestic'] = h4_df['transferSource_countryId'] == h4_df['transferDestination_countryId']
h4_df['is_upward'] = h4_df['source_tier_num'] > h4_df['dest_tier_num']
h4_df['is_lower_source'] = h4_df['source_tier_num'] >= 2
h4_df['is_top_source'] = h4_df['source_tier_num'] == 1

h4_df[['source_tier_num','dest_tier_num','is_domestic','is_upward']].head()


In [ ]:
# Tier-flow matrix (share)
tier_flow = pd.crosstab(
    h4_df['source_tier_num'],
    h4_df['dest_tier_num'],
    normalize='index'
)
tier_flow.round(3).iloc[:10, :10]


In [ ]:
# Lower-tier source: domestic and upward shares
lower_df = h4_df[h4_df['is_lower_source']].copy()

lower_stats = pd.DataFrame({
    'n_transfers_lower_source': [len(lower_df)],
    'domestic_share_lower_source': [lower_df['is_domestic'].mean() if len(lower_df) else None],
    'upward_share_lower_source': [lower_df['is_upward'].mean() if len(lower_df) else None],
    'cross_border_share_lower_source': [1 - lower_df['is_domestic'].mean() if len(lower_df) else None],
}).round(4)

lower_stats


In [ ]:
# Compare lower-tier vs top-tier source patterns
top_df = h4_df[h4_df['is_top_source']].copy()

comparison = pd.DataFrame({
    'group': ['lower_source', 'top_source'],
    'n_transfers': [len(lower_df), len(top_df)],
    'domestic_share': [
        lower_df['is_domestic'].mean() if len(lower_df) else None,
        top_df['is_domestic'].mean() if len(top_df) else None,
    ],
    'cross_border_share': [
        1 - lower_df['is_domestic'].mean() if len(lower_df) else None,
        1 - top_df['is_domestic'].mean() if len(top_df) else None,
    ],
    'upward_share': [
        lower_df['is_upward'].mean() if len(lower_df) else None,
        top_df['is_upward'].mean() if len(top_df) else None,
    ],
}).round(4)

comparison


## 07 ? Hypothesis Tests & Robustness (stub)

In [ ]:
# TODO: H1?H4 tests + with/without loans + per-season vs aggregated
pass

In [ ]:
# Plotly renderer setup (independent from Jupyter mime stack)
import plotly.io as pio
pio.renderers.default = 'browser'
pio.renderers.default


## 08 ? Visuals (stub)

In [ ]:
# RQ3: Community-first network visualization (interactive)
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

plot_edges = edge_all.sort_values('n_transfers', ascending=False).head(500).copy()
nodes_used = sorted(set(plot_edges['source_league_id']).union(set(plot_edges['dest_league_id'])))
m_plot = metrics[metrics['league'].isin(nodes_used)].copy()
m_plot = m_plot.dropna(subset=['community']).copy()
m_plot['community'] = m_plot['community'].astype(int)

# Keep only edges where both nodes have community labels
valid_nodes = set(m_plot['league'])
plot_edges = plot_edges[plot_edges['source_league_id'].isin(valid_nodes) & plot_edges['dest_league_id'].isin(valid_nodes)].copy()

# Community centers on a circle
communities = sorted(m_plot['community'].unique().tolist())
n_comm = len(communities)
angles = np.linspace(0, 2 * np.pi, n_comm, endpoint=False)
comm_center = {c: (8 * np.cos(a), 8 * np.sin(a)) for c, a in zip(communities, angles)}

# Place nodes within each community in local radial layout
pos = {}
for c in communities:
    g = m_plot[m_plot['community'] == c].sort_values('in_strength', ascending=False).copy()
    n = len(g)
    if n == 1:
        row = g.iloc[0]
        pos[row['league']] = comm_center[c]
        continue
    local_angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    for rank, (_, row) in enumerate(g.iterrows()):
        r = 0.8 + 2.2 * (rank / max(1, n - 1))
        cx, cy = comm_center[c]
        x = cx + r * np.cos(local_angles[rank])
        y = cy + r * np.sin(local_angles[rank])
        pos[row['league']] = (x, y)

palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
comm_color = {c: palette[i % len(palette)] for i, c in enumerate(communities)}

# Build edge traces (split intra/inter-community for readability)
m_map = m_plot.set_index('league')
intra_x, intra_y, inter_x, inter_y = [], [], [], []
for r in plot_edges.itertuples(index=False):
    s, t = r.source_league_id, r.dest_league_id
    if s not in pos or t not in pos:
        continue
    x0, y0 = pos[s]
    x1, y1 = pos[t]
    same = int(m_map.loc[s, 'community']) == int(m_map.loc[t, 'community'])
    if same:
        intra_x += [x0, x1, None]
        intra_y += [y0, y1, None]
    else:
        inter_x += [x0, x1, None]
        inter_y += [y0, y1, None]

edge_intra = go.Scatter(
    x=intra_x, y=intra_y, mode='lines', hoverinfo='none',
    line=dict(width=0.8, color='rgba(100,100,100,0.35)')
)
edge_inter = go.Scatter(
    x=inter_x, y=inter_y, mode='lines', hoverinfo='none',
    line=dict(width=0.5, color='rgba(160,160,160,0.18)')
)

# Nodes
node_x, node_y, node_color, node_size, node_text = [], [], [], [], []
for _, row in m_plot.iterrows():
    lid = row['league']
    x, y = pos[lid]
    node_x.append(x)
    node_y.append(y)
    node_color.append(comm_color[int(row['community'])])
    node_size.append(max(8, np.log1p(float(row['in_strength'])) * 4))
    node_text.append(
        f"{row['league_name']}<br>ID: {lid}<br>Community: {int(row['community'])}<br>In-strength: {float(row['in_strength']):.0f}<br>Betweenness: {float(row['betweenness']):.2f}"
    )

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers', hoverinfo='text', text=node_text,
    marker=dict(size=node_size, color=node_color, line=dict(width=0.7, color='white'))
)

fig_net = go.Figure(data=[edge_inter, edge_intra, node_trace])
fig_net.update_layout(
    title='RQ3 Community Network (community-first layout)',
    showlegend=False,
    template='plotly_white',
    margin=dict(l=10, r=10, t=50, b=10),
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False),
)
fig_net.show(renderer='browser')


### RQ4 Pathway Sankey
Tier-to-tier flow visualization for lower-tier pathways.

In [ ]:
import plotly.graph_objects as go

# Choose scope: 'all', 'domestic', or 'international'
scope = 'all'
if scope == 'domestic':
    sankey_df = h4_df[h4_df['is_domestic']].copy()
elif scope == 'international':
    sankey_df = h4_df[~h4_df['is_domestic']].copy()
else:
    sankey_df = h4_df.copy()

flows = (sankey_df.groupby(['source_tier_num','dest_tier_num'], dropna=False)
    .size().reset_index(name='n'))

source_labels = [f"Src T{int(t)}" for t in sorted(flows['source_tier_num'].unique())]
dest_labels = [f"Dst T{int(t)}" for t in sorted(flows['dest_tier_num'].unique())]
labels = source_labels + dest_labels

src_map = {t: i for i, t in enumerate(sorted(flows['source_tier_num'].unique()))}
dst_map = {t: i + len(source_labels) for i, t in enumerate(sorted(flows['dest_tier_num'].unique()))}

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(label=labels, pad=12, thickness=14),
    link=dict(
        source=[src_map[t] for t in flows['source_tier_num']],
        target=[dst_map[t] for t in flows['dest_tier_num']],
        value=flows['n'].tolist(),
    )
)])
fig_sankey.update_layout(
    title=f'RQ4 Tier Pathways ({scope})',
    template='plotly_white',
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_sankey.show(renderer='browser')


## 09 ? Results Summary (stub)

In [ ]:
# TODO: final tables + short interpretation
pass